# Public PIV measurement comparison

This optional notebook compares the installed recommended model with the hub-height PIV dataset by **Majid Bastankhah and Fernando Porté-Agel**, [Zenodo 3607598, v1](https://zenodo.org/records/3607598), DOI 10.5281/zenodo.3607598. The dataset accompanies their 2016 JFM study, *Experimental and theoretical study of wind turbine wakes in yawed conditions*. Download `Bastankhah_jfm_2016.h5` from that record and inspect its data-use terms. No measurement files are bundled here and no automatic download occurs.

Install the notebook and validation extras. Set `WAKEMARCH_PIV_DATA` to the downloaded file, or edit `DATA` below. Missing data raises an error rather than substituting a synthetic demonstration. The published MD5 is checked before reading; SHA-256 is recorded for reproducibility.

**Calibration disclosure:** CT=0.9 and wake-growth k=0.024 are inherited from the authors' unyawed calibration in the accompanying study. TI is the equivalent value obtained by inverting the accepted k(TI) relation, not an independently measured ambient TI. No fitting occurs in this notebook. These yaw cases have been examined during model development; this is an illustrative measurement comparison, not a new blind validation or a reproduction of every manuscript metric.

This comparison deliberately retains its original fixed CT, inflow and calibration protocol. Do not substitute turbine curves or the fixed-coefficient power estimate when reproducing these PIV comparisons; they are separate operating assumptions.

In [ ]:
import os, hashlib, json, sys
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
import wakemarch
from wakemarch import Numerics, WakeModel

DATA = Path(os.environ.get("WAKEMARCH_PIV_DATA", "Bastankhah_jfm_2016.h5")).expanduser()
if not DATA.is_file():
    raise FileNotFoundError("Download Zenodo 3607598 and set DATA or WAKEMARCH_PIV_DATA")
payload = DATA.read_bytes()
assert hashlib.md5(payload).hexdigest() == "00d23d90751e476569b5ad186513d3a4", "File differs from the pinned Zenodo record"
source_sha256 = hashlib.sha256(payload).hexdigest()
del payload
print(wakemarch.__version__, wakemarch.__file__, source_sha256)

## Coordinates, parameters and comparison support

The file has normalized x/D, y/D and u/U arrays. We use the study's H/D=0.125/0.15 with equivalent model scales D=1 m and U=1 m/s; plotted values are nondimensional. Data positive-y deflection is mapped to WakeMarch negative-y by `y_model = -y_data`. This sign is fixed, not fitted separately for each case.

For each yaw, select the nearest measured x plane to 3D, 6D and 10D. Sample the model at that **actual** x and each measured lateral coordinate. Compare only finite measurements within |y|≤1.2D. No interpolation of measurements, smoothing or thresholding is used. The pooled RMS velocity discrepancy is normalized by U (already one here); it has no uncertainty weighting and is not identical to the historical scoreboard metrics. This hub-height comparison does not validate cross-plane curl.

In [ ]:
records, profiles = [], {}
TI_equivalent = (0.024 - 0.003678)/0.3837
with h5py.File(DATA, "r") as dataset:
    for yaw in (10, 20, 30):
        group = dataset[f"Yaw_{yaw}"]
        xs, ys = group["u/x"][0], -group["u/y"][0]
        measured = group["u/avg"][:]
        assert measured.shape == (len(ys), len(xs))
        params = dict(D=1., H=0.125/0.15, U=1., CT=0.9, yaw_deg=yaw, TI=TI_equivalent)
        model = WakeModel(**params, numerics=Numerics(h_D=1/20))
        result = model.solve(10, progress=True)
        errors, planes = [], []
        for target in (3, 6, 10):
            i = int(np.argmin(abs(xs-target)))
            keep = (abs(ys)<=1.2) & np.isfinite(measured[:, i])
            assert keep.sum() > 20
            y = ys[keep]
            observed = measured[keep, i]
            predicted = result.sample_streamwise_velocity(float(xs[i]), y, model.H)
            assert np.isfinite(predicted).all()
            errors.extend((predicted-observed).tolist())
            planes.append(dict(x_D=float(xs[i]), points=int(keep.sum())))
            if target == 6:
                profiles[yaw] = (y, observed, predicted, float(xs[i]))
        records.append(dict(yaw_deg=yaw, parameters=params, h_D=1/20,
            planes=planes, rms_velocity_difference_over_U=float(np.sqrt(np.mean(np.square(errors))))))
print(json.dumps(dict(version=wakemarch.__version__, source_sha256=source_sha256,
    source_doi="10.5281/zenodo.3607598", calibration="inherited unyawed CT and k; TI is equivalent", results=records), indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, yaw in zip(axes, (10, 20, 30)):
    y, observed, predicted, x = profiles[yaw]
    order = np.argsort(y)
    ax.plot(y[order], 1-observed[order], '.', label='PIV')
    ax.plot(y[order], 1-predicted[order], '-', label='WakeMarch')
    ax.set(xlabel='y / D (model frame)', title=f'{yaw}° yaw, x={x:.3f}D')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Deficit / U')
axes[0].legend()
fig.tight_layout()
plt.show()

Change numerical resolution to examine discretization sensitivity, keeping calibration and comparison support fixed. Report failures and discrepancies rather than retuning to make this example pass. The assertions check file identity and pipeline validity, **not** a prescribed predictive-accuracy threshold. The accepted recommended model and historical scientific caveats still apply.